# muon-encoder pilot: throughput on this GPU

**Use a fresh notebook** (File → Import into a new one), or Run → Factory reset first: a kernel that ran the old smoke test still holds the GPU and `train.py` will OOM.

Settings: **Internet on**, **Accelerator: GPU T4 x2** (uses one). **Run All**. ~10 minutes.

Writes the `muon-encoder` code into the session, tokenises a small FineWeb-Edu slice, then trains the 17M model for 100 steps under Muon and under AdamW in fp16. The number to read is **tok/s**.

In [ ]:
!pip install -q -U transformers datasets wandb 2>&1 | tail -1
%env WANDB_MODE=offline
%env WANDB_SILENT=true
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv

In [ ]:
%%writefile config.py
from transformers import ModernBertConfig

LADDER = {
    "17m": dict(num_hidden_layers=7, hidden_size=256, intermediate_size=384, num_attention_heads=4),
    "32m": dict(num_hidden_layers=10, hidden_size=384, intermediate_size=576, num_attention_heads=6),
    "68m": dict(num_hidden_layers=19, hidden_size=512, intermediate_size=768, num_attention_heads=8),
    "150m": dict(num_hidden_layers=22, hidden_size=768, intermediate_size=1152, num_attention_heads=12),
}

BASE_WIDTH = 256

VOCAB_SIZE = 50368
CLS_ID, SEP_ID, PAD_ID, MASK_ID = 50281, 50282, 50283, 50284


def make_config(size, seq_len):
    return ModernBertConfig(
        vocab_size=VOCAB_SIZE,
        max_position_embeddings=seq_len,
        global_attn_every_n_layers=3,
        local_attention=128,
        global_rope_theta=160000.0,
        local_rope_theta=160000.0,
        attention_bias=False,
        mlp_bias=False,
        norm_bias=False,
        classifier_bias=False,
        decoder_bias=True,
        tie_word_embeddings=True,
        sparse_prediction=True,
        pad_token_id=PAD_ID,
        bos_token_id=CLS_ID,
        eos_token_id=SEP_ID,
        cls_token_id=CLS_ID,
        sep_token_id=SEP_ID,
        **LADDER[size],
    )

In [ ]:
%%writefile data.py
import numpy as np
import torch
from config import VOCAB_SIZE, MASK_ID


class TokenStream:
    def __init__(self, path, seq_len, batch_size):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.pos = 0

    def next_batch(self):
        n = self.seq_len * self.batch_size
        if self.pos + n > len(self.data):
            self.pos = 0
        chunk = self.data[self.pos : self.pos + n].astype(np.int64)
        self.pos += n
        return torch.from_numpy(chunk).view(self.batch_size, self.seq_len)


def mask_tokens(tokens, mask_rate, generator):
    r = torch.rand(tokens.shape, generator=generator)
    masked = r < mask_rate
    labels = tokens.masked_fill(~masked, -100)
    inputs = tokens.clone()
    inputs[r < 0.8 * mask_rate] = MASK_ID
    random_pos = (r >= 0.8 * mask_rate) & (r < 0.9 * mask_rate)
    random_ids = torch.randint(VOCAB_SIZE, tokens.shape, generator=generator)
    inputs[random_pos] = random_ids[random_pos]
    return inputs, labels

In [ ]:
%%writefile muon.py
import math
import torch

NS_COEFFS = (3.4445, -4.7750, 2.0315)
NS_STEPS = 5
QUANTILES = (0.1, 0.25, 0.5, 0.75, 0.9)


def ns_polynomial(x, coeffs=NS_COEFFS, steps=NS_STEPS):
    a, b, c = coeffs
    for _ in range(steps):
        x = a * x + b * x**3 + c * x**5
    return x


def ns_threshold(target=0.1):
    lo, hi = 1e-6, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if ns_polynomial(mid) < target else (lo, mid)
    return hi


NS_THRESHOLD = ns_threshold()


def newton_schulz(G, dtype=torch.bfloat16, coeffs=NS_COEFFS, steps=NS_STEPS):
    a, b, c = coeffs
    X = G.to(dtype)
    X = X / (X.norm() + 1e-7)
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


def spectral_quantiles(G):
    s = torch.linalg.svdvals(G.float() / G.norm())
    r = s.numel()
    out = {f"sv_q{q}": s[math.ceil(q * r) - 1].item() for q in QUANTILES}
    out["frac_below_threshold"] = (s < NS_THRESHOLD).float().mean().item()
    return out


class Muon(torch.optim.Optimizer):
    def __init__(
        self, params, lr=0.02, momentum=0.95, weight_decay=0.0, nesterov=True, ns_dtype=torch.bfloat16
    ):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=nesterov)
        super().__init__(params, defaults)
        self.ns_dtype = ns_dtype
        self.spectra = {}

    @torch.no_grad()
    def step(self, track_spectra=False):
        self.spectra = {}
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p)
                buf = state["momentum_buffer"]
                buf.lerp_(p.grad, 1 - group["momentum"])
                g = p.grad.lerp(buf, group["momentum"]) if group["nesterov"] else buf
                if track_spectra:
                    self.spectra[p] = spectral_quantiles(g)
                update = newton_schulz(g, self.ns_dtype) * max(1, p.size(0) / p.size(1)) ** 0.5
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(update, alpha=-group["lr"])

In [ ]:
%%writefile prepare_data.py
import argparse
import os
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

parser = argparse.ArgumentParser()
parser.add_argument("--train_tokens", type=float, default=3e9)
parser.add_argument("--val_tokens", type=float, default=2e7)
parser.add_argument("--out_dir", default="data")
parser.add_argument("--docs_per_batch", type=int, default=1000)
args = parser.parse_args()

os.makedirs(args.out_dir, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
tokenizer.model_max_length = int(1e9)
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True
)
docs = iter(dataset)


def write_split(name, budget):
    path = os.path.join(args.out_dir, f"{name}.bin")
    written = 0
    with open(path, "wb") as f:
        while written < budget:
            texts = [next(docs)["text"] for _ in range(args.docs_per_batch)]
            ids = tokenizer(texts)["input_ids"]
            flat = np.concatenate([np.array(x, dtype=np.uint16) for x in ids])
            f.write(flat.tobytes())
            written += len(flat)
            print(f"\r{name}: {written / 1e6:.1f}M tokens", end="", flush=True)
    print()


write_split("val", args.val_tokens)
write_split("train", args.train_tokens)
os._exit(0)

In [ ]:
%%writefile train.py
import argparse
import json
import math
import os
import sys
import time
import torch
import wandb
from transformers import ModernBertForMaskedLM
from config import make_config, BASE_WIDTH
from data import TokenStream, mask_tokens
from muon import Muon

parser = argparse.ArgumentParser()
parser.add_argument("--size", default="17m")
parser.add_argument("--treatment", default="muon", choices=["muon", "adamw", "mup"])
parser.add_argument("--lr", type=float, default=0.02)
parser.add_argument("--adam_lr", type=float, default=1e-3)
parser.add_argument("--wd", type=float, default=0.1)
parser.add_argument("--momentum", type=float, default=0.95)
parser.add_argument("--betas", type=float, nargs=2, default=(0.9, 0.95))
parser.add_argument("--tokens_per_param", type=float, default=20)
parser.add_argument("--batch_size", type=int, default=64)
parser.add_argument("--seq_len", type=int, default=512)
parser.add_argument("--mask_rate", type=float, default=0.3)
parser.add_argument("--warmup_frac", type=float, default=0.05)
parser.add_argument("--grad_clip", type=float, default=1.0)
parser.add_argument("--seed", type=int, default=0)
parser.add_argument("--eval_every", type=int, default=500)
parser.add_argument("--eval_batches", type=int, default=20)
parser.add_argument("--final_eval_batches", type=int, default=200)
parser.add_argument("--spectral_every", type=int, default=20)
parser.add_argument("--log_every", type=int, default=20)
parser.add_argument("--ckpt_every", type=int, default=500)
parser.add_argument("--max_steps", type=int, default=None)
parser.add_argument("--data_dir", default="data")
parser.add_argument("--out_dir", default="runs")
parser.add_argument("--wandb_project", default="muon-encoder")
parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
parser.add_argument("--dtype", default="bf16", choices=["bf16", "fp16", "fp32"])
parser.add_argument("--compile", action="store_true")
args = parser.parse_args()

run_name = (
    f"{args.size}-{args.treatment}-lr{args.lr}-wd{args.wd}"
    f"-tpp{args.tokens_per_param:g}-s{args.seed}"
)
run_dir = os.path.join(args.out_dir, run_name)
results_path = os.path.join(run_dir, "results.json")
ckpt_path = os.path.join(run_dir, "ckpt.pt")
if os.path.exists(results_path):
    print(f"{run_name} already finished")
    sys.exit()
os.makedirs(run_dir, exist_ok=True)

torch.manual_seed(args.seed)
device = args.device
dtype = dict(bf16=torch.bfloat16, fp16=torch.float16, fp32=torch.float32)[args.dtype]
autocast = torch.autocast(device.split(":")[0], dtype=dtype, enabled=args.dtype != "fp32")
scaler = torch.amp.GradScaler(enabled=args.dtype == "fp16")

config = make_config(args.size, args.seq_len)
model = ModernBertForMaskedLM(config).to(device)
raw_model = model
param_names = {p: n for n, p in model.named_parameters()}
n_params = sum(p.numel() for p in model.parameters())
n_body = sum(p.numel() for n, p in model.named_parameters() if p.ndim == 2 and "layers." in n)


def build_optimizers(model):
    body = [p for n, p in model.named_parameters() if p.ndim == 2 and "layers." in n]
    rest = [p for n, p in model.named_parameters() if not (p.ndim == 2 and "layers." in n)]
    lr, wd = args.lr, args.wd
    if args.treatment == "mup":
        width_ratio = BASE_WIDTH / config.hidden_size
        lr, wd = lr * math.sqrt(width_ratio), wd * width_ratio
    if args.treatment == "adamw":
        groups = [dict(params=body, weight_decay=wd), dict(params=rest, weight_decay=0.0)]
        return None, torch.optim.AdamW(groups, lr=lr, betas=args.betas)
    ns_dtype = torch.float16 if args.dtype == "fp16" else torch.bfloat16
    muon = Muon(body, lr=lr, momentum=args.momentum, weight_decay=wd, ns_dtype=ns_dtype)
    adam = torch.optim.AdamW(rest, lr=args.adam_lr, betas=args.betas, weight_decay=0.0)
    return muon, adam


muon, adam = build_optimizers(model)
optimizers = [opt for opt in (muon, adam) if opt is not None]
for opt in optimizers:
    for group in opt.param_groups:
        group["base_lr"] = group["lr"]

tokens_per_step = args.batch_size * args.seq_len
total_steps = math.ceil(args.tokens_per_param * n_params / tokens_per_step)
if args.max_steps is not None:
    total_steps = min(total_steps, args.max_steps)
warmup_steps = int(args.warmup_frac * total_steps)


def lr_multiplier(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))


train_stream = TokenStream(os.path.join(args.data_dir, "train.bin"), args.seq_len, args.batch_size)
val_stream = TokenStream(os.path.join(args.data_dir, "val.bin"), args.seq_len, args.batch_size)
mask_gen = torch.Generator().manual_seed(args.seed)

start_step = 0
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    for opt, state in zip(optimizers, ckpt["optimizers"]):
        opt.load_state_dict(state)
    scaler.load_state_dict(ckpt["scaler"])
    train_stream.pos = ckpt["stream_pos"]
    mask_gen.set_state(ckpt["mask_gen"])
    start_step = ckpt["step"] + 1
    print(f"resuming {run_name} from step {start_step}")

if args.compile:
    model = torch.compile(model)


def save_checkpoint(step):
    torch.save(
        dict(
            model=raw_model.state_dict(),
            optimizers=[opt.state_dict() for opt in optimizers],
            scaler=scaler.state_dict(),
            stream_pos=train_stream.pos,
            mask_gen=mask_gen.get_state(),
            step=step,
        ),
        ckpt_path,
    )


@torch.no_grad()
def evaluate(n_batches):
    model.eval()
    val_stream.pos = 0
    gen = torch.Generator().manual_seed(1234)
    losses = []
    for _ in range(n_batches):
        inputs, labels = mask_tokens(val_stream.next_batch(), args.mask_rate, gen)
        with autocast:
            loss = model(input_ids=inputs.to(device), labels=labels.to(device)).loss
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def log_spectra(step, spectra_file):
    for p, quantiles in muon.spectra.items():
        layer = param_names[p].replace("model.layers.", "").replace(".weight", "")
        spectra_file.write(json.dumps(dict(step=step, layer=layer, **quantiles)) + "\n")
        wandb.log({f"spectra/{layer}/{k}": v for k, v in quantiles.items()}, step=step)


wandb.init(
    project=args.wandb_project,
    name=run_name,
    id=run_name.replace(".", "p"),
    resume="allow",
    config=vars(args),
)
wandb.config.update(dict(n_params=n_params, n_body_params=n_body, total_steps=total_steps))
print(f"{run_name}: {n_params / 1e6:.1f}M params ({n_body / 1e6:.1f}M body), {total_steps} steps")
spectra_path = os.path.join(run_dir, "spectra.jsonl")
if start_step > 0 and os.path.exists(spectra_path):
    kept = [line for line in open(spectra_path) if json.loads(line)["step"] < start_step]
    open(spectra_path, "w").writelines(kept)
spectra_file = open(spectra_path, "a")
model.train()
start = time.time()

for step in range(start_step, total_steps):
    mult = lr_multiplier(step)
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["base_lr"] * mult

    inputs, labels = mask_tokens(train_stream.next_batch(), args.mask_rate, mask_gen)
    with autocast:
        loss = model(input_ids=inputs.to(device), labels=labels.to(device)).loss
    scaler.scale(loss).backward()
    for opt in optimizers:
        scaler.unscale_(opt)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

    track = muon is not None and step % args.spectral_every == 0
    if muon is not None:
        scaler.step(muon, track_spectra=track)
    scaler.step(adam)
    scaler.update()
    model.zero_grad(set_to_none=True)

    if track:
        log_spectra(step, spectra_file)
    if step % args.log_every == 0:
        elapsed = time.time() - start
        tokens_per_sec = (step + 1 - start_step) * tokens_per_step / elapsed
        wandb.log(
            dict(
                train_loss=loss.item(),
                grad_norm=grad_norm.item(),
                lr_mult=mult,
                tokens=(step + 1) * tokens_per_step,
                tokens_per_sec=tokens_per_sec,
            ),
            step=step,
        )
        print(f"step {step}/{total_steps} loss {loss.item():.4f} {tokens_per_sec:,.0f} tok/s")
    if step % args.eval_every == 0 and step > 0:
        wandb.log(dict(val_loss=evaluate(args.eval_batches)), step=step)
    if step % args.ckpt_every == 0 and step > 0:
        save_checkpoint(step)

final_val_loss = evaluate(args.final_eval_batches)
wandb.log(dict(val_loss=final_val_loss, final_val_loss=final_val_loss), step=total_steps)
spectra_file.close()
results = dict(
    run_name=run_name,
    final_val_loss=final_val_loss,
    n_params=n_params,
    n_body_params=n_body,
    tokens=total_steps * tokens_per_step,
    steps=total_steps,
    wall_time=time.time() - start,
    args=vars(args),
)
json.dump(results, open(results_path, "w"), indent=2)
print(f"final val loss {final_val_loss:.4f}")
wandb.finish()

In [ ]:
!python prepare_data.py --train_tokens 5e6 --val_tokens 1e6

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!python train.py --size 17m --treatment muon --dtype fp16 --max_steps 100 --final_eval_batches 10 --spectral_every 20 --log_every 10 --out_dir /kaggle/working/pilot

In [ ]:
!python train.py --size 17m --treatment adamw --lr 1e-3 --dtype fp16 --max_steps 100 --final_eval_batches 10 --log_every 10 --out_dir /kaggle/working/pilot

In [ ]:
import json, glob
for path in glob.glob("/kaggle/working/pilot/*/results.json"):
    r = json.load(open(path))
    print(f"{r['run_name']:40s} {r['tokens'] / r['wall_time']:,.0f} tok/s  val loss {r['final_val_loss']:.3f}")

The `wall_time` figure includes the final eval, so the steady-state tok/s printed during training is the better number. Paste this notebook back and I'll rebuild the compute budget from it.